# FASE 02 — EDA Recap: Synthese nb05 · nb06 · nb07
**Masterproef:** Tier-Stratified Occupancy Prediction and Scenario-Based Policy
Simulation in Urban Parking Systems — A Case Study of Mechelen
**Auteur:** Emile Van de Voorde
**Status:** Exploratieve Data-Analyse volledig afgerond — input voor nb08

---

## 1. Inleiding en scope

Deze recap synthethiseert de bevindingen uit drie exploratieve
notebooks (nb05–nb07) die samen de empirische basis vormen voor de
feature engineering (nb08) en modellering (nb09–nb12) van deze
masterproef. De centrale onderzoeksvraag — in welke mate externe
factoren de parkeeroccupantie beïnvloeden en of dit effect
tier-specifiek is — wordt exploratief getoetst via 17 falsifieerbare
hypothesen (H-T1–H-T4, H-S1–H-S4, H-E1–H-E9).

> **Methodologische kanttekening:** Alle bevindingen in nb05–nb07
> zijn *associatief* van aard. Observationele tijdreeksdata laat geen
> causale inferentie toe zonder experimentele controle (James et al.,
> 2021, p. 59). De formulering "geassocieerd met" is in alle
> conclusies gehanteerd.

**Trainingsset:** 129.837 rijen (2020, 2023, 2024) na kwaliteitsfilter
(low_data_coverage=0, system_blackout=0, partial_year=0).
**Holdout:** 2025 (87.600 rijen, temporeel geïsoleerd).
**Tiers:** centrum (5 parkings), vesten_of_rand (4+1 parkings).

---
# Zeven Structurele Blinde Vlekken (notebook-overschrijdend)

Dit zijn risico's die nergens expliciet worden benoemd, maar meerdere bevindingen tegelijk raken.

1. Het 2020-trainingsdata-dilemma
r_struct = −0.956 impliceert dat 2020-patronen structureel tegengesteld zijn aan 2023/2024. Inclusie van 2020 in de trainingsset is een expliciete modelleeringskeuze die onderbouwing vereist. Een gevoeligheidsanalyse (model met vs. zonder 2020) moet in nb09 worden uitgevoerd en gerapporteerd.

2. event-types

| Feature | Prioriteit | Motivatie |
|---|---|---|
| `is_kermis` | 🔴 HOOG | Medium effect beide tiers, n ≈ 5.700, stadswijd |
| `is_voetbal × tier` | 🔴 HOOG | Tegengesteld effect bevestigd, interactie verplicht |
| `is_festival × tier` | 🔴 HOOG *(revisie)* | Medium effect, centrum negatief, t-toets was onderschatting |
| `is_processie` | 🟡 MEDIUM | Significant maar klein, n beperkt |
| `is_carnaval × tier` | 🟡 MEDIUM | Gefaseerd effect, inhoudelijk gemotiveerd |

3. De systematische Fischer z-bug


4. Feature leakage bij lag-features op de holdout
occ_lag_168h vereist bezettingsdata van één week eerder. Voor de eerste week van de holdout (2025) zijn die lag-waarden afkomstig uit de trainingsperiode — dat is correct. Maar is dit ook zo geïmplementeerd? Een fout hier (waarbij lag-waarden deels uit de toekomst komen) is een klassieke data leakage-bug die prestaties kunstmatig opblaast. Expliciete documentatie van de lag-constructie is vereist in nb08.

5. Confound daglicht bij temperatuureffect
   Correctie:

| Feature                     | Prioriteit vóór check | Prioriteit ná check    | Reden                                              |
| --------------------------- | --------------------- | ---------------------- | -------------------------------------------------- |
| temp_c                      | 🟡 MEDIUM             | 🟢 LAAG (of schrappen) | Globaal ρ ≈ 0; seizoenspatroon al in month_sin/cos |
| temp_c × seizoen interactie | 🟡 MEDIUM             | 🔴 VERWIJDEREN         | Seizoenseffect is de confound, niet de moderator   |
| sun_duration_min            | 🟢 LAAG               | 🟢 LAAG (bevestigd)    | ρ = +0.081, maar collineair met temp — consistent  |
| month_sin + month_cos       | 🔴 HOOG               | 🔴 HOOG (versterkt)    | Vangt het dominante seizoenspatroon volledig op    |

6. De bingrenzen van precip_bin zijn niet verantwoord
Nergens gedocumenteerd. Noodzakelijk voor reproduceerbaarheid en methodologische verantwoording in de thesis.

7. Ontbrekende alternatieve tierclassificatie
Zie laatste cellen nb06 De administratieve tierindeling is nooit vergeleken met een data-gedreven clustering. Gegeven de zwakke H-S3-resultaten is dit een legitieme methodologische lacune. Minimumvereiste: vermeld het als beperking en als aanbeveling voor vervolgonderzoek.

## Aannames
#### 1. LT-data en event confidence uitsluiten

Long-term data: akkoord, maar met nuancering

Het uitsluiten van LT-data als target is methodologisch de correcte beslissing, en ze is goed te verantwoorden. Hieronder de academische redenering.

Academische formulering voor de thesis:

"De short-term (ST) en long-term (LT) bezettingsdata worden in dit onderzoek gescheiden gemodelleerd. De LT-data, afkomstig van abonnementhouders met habitueel en nagenoeg constant rijgedrag, vertoont structureel andere variantiepatronen dan het transiente ST-gebruik dat centraal staat in de onderzoeksvragen (Lira et al., 2021). De target-variabele van het predictiemodel is bijgevolg uitsluitend de ST-bezettingsgraad (occupancy_rate uit MAD_shortterm.parquet). LT-bezetting wordt niet als target opgenomen, maar kan in nb08 worden geëvalueerd als contextuele feature voor parkings waar het empirisch is vastgesteld dat LT-druk de beschikbare ST-ruimte structureel beperkt (P Hoogstraat, P Komet — zie nb07b, Sectie 2.3)."

Kanttekening die je niet moet vergeten: voor P Hoogstraat en P Komet geldt een ceiling effect: LT neemt respectievelijk ~72% en ~72% van de capaciteit in. De ST-bezettingsgraad is er fundamenteel beperkt in variabiliteit (maximum bereikbaar is ~28%). Dit verlaagt de theoretische voorspelbaarheid voor die parkings. Vermeld dit expliciet als een per-parking data-limitatie — niet als een reden om die parkings uit te sluiten, maar als context bij het interpreteren van hogere MAPE-waarden die je er waarschijnlijk zal zien in nb11.

#### 2. Event confidence: gedeeltelijk akkoord — nuance is hier essentieel

Volledig uitsluiten van alle event-features is een valide maar onomkeerbare beslissing. De genuanceerde academische positie is sterker:

Optie A — Volledig uitsluiten (wat jij overweegt):

"Gegeven dat 53,5% van de event-observaties (185/346) een data_confidence = 'estimated' heeft — wat betekent dat tijdstip, duur en/of schaal niet empirisch zijn geverifieerd — wordt geconcludeerd dat de event-feature-set als geheel niet voldoet aan de meetbetrouwbaarheidsnormen die een ML-model vereist voor stabiele parameterschatting. Event-features worden bijgevolg niet opgenomen in de feature engineering pipeline (nb08). Dit is een expliciete databegrenzingsbeslissing: de bevindingen van H-S4 en H-E3 blijven relevant als exploratieve observaties, maar zijn onvoldoende robuust om te vertalen naar predictieve features."

Optie B — Alleen verified events behouden (methodologisch sterker):
Behoud de ~161 geverifieerde event-dagen als binaire dummies. De cascade-features (hours_to_next_event) worden dan alleen berekend op basis van verified events. Dit is wetenschappelijk eerlijker én haalt meer signaal eruit.

Mijn aanbeveling: Als je H-E3 (cascade) al bevestigd hebt, gooi je potentieel krachtige features weg bij Optie A. Optie B is methodologisch verdedigbaarder. Maar als je de administratieve last van event-verificatie te groot vindt voor de resterende thesis-tijd, dan is Optie A verdedigbaar mits je het expliciet rapporteert als bovenstaand.


#### 3. Waar en wanneer rapporteer je de 2020-gevoeligheidsanalyse en de lag-leakage beslissing?

2020-gevoeligheidsanalyse

Wanneer: In nb09 (baseline modellen), onmiddellijk na de eerste modeltraining.

Hoe: Train hetzelfde baseline model (bijv. XGBoost of Random Forest) twee keer:

Run A: trainingsdata = 2020 + 2023 + 2024

Run B: trainingsdata = 2023 + 2024 only

Valideer beide runs op 2024 (temporele cross-validatie) via MAPE of RMSE. Als Run B significant beter valideert, is 2020-data contraproductief en gebruik je alleen 2023+2024 als definitieve trainingsset.

Academische formulering voor de thesis (methodologie-sectie):

"Gezien de structureel afwijkende patronen in de COVID-periode 2020 (H-T4, r_struct = −0.956 voor centrumtier), werd een gevoeligheidsanalyse uitgevoerd waarbij het model werd getraind met (a) 2020 + 2023 + 2024 en (b) 2023 + 2024 uitsluitend. De validatieprestaties op het jaar 2024 bepaalden de definitieve keuze van de trainingsperiode."

#### 4.Lag-leakage: eerste week 2025 (occ_lag_168h)

Wanneer: In nb08 (feature engineering), in de cel die lag-features construeert.

Wat documenteren: Bij de constructie van occ_lag_168h voor de holdout (2025) zijn de eerste 168 uur (= week 1 van januari 2025) per definitie afhankelijk van trainingsdata (december 2024). Dit is correct en geen leakage. Maar vermeld expliciet:

Dat de lag-waarden voor de eerste week van de holdout zijn geïmporteerd vanuit de laatste week van de trainingsset

Dat er geen look-ahead leakage is (toekomstige bezetting wordt nooit gebruikt)

Dat je de eerste 168 uur van de holdout expliciet hebt gecontroleerd op NaN-waarden

Academische formulering (nb08, markdown-cel):

"Tijdreeksintegriteit: occ_lag_168h voor de holdoutperiode (2025) vereist bezettingswaarden uit de laatste week van de trainingsperiode (december 2024). Deze waarden zijn aaneengesloten beschikbaar en bevatten geen missing data na de kwaliteitsfilter. Er treedt geen temporele leakage op: geen enkel lag-feature bevat bezettingsinformatie uit de toekomst ten opzichte van het voorspelmoment. De eerste 168 observaties van de holdout worden gecontroleerd op NaN-completeness vóór modelinferentie."

#### 5. Steekproefomvang per event-type & daglengte-confound dubbelchecken

- Steekproefomvang per event-type
- Daglengte confound
  

___

## 2. nb05 — Temporele EDA

### 2.1 Hypotheseresultaten

#### H-T1 — Dagelijks bezettingsprofiel
*Verwachting:* Bimodaal weekdagprofiel (ochtend + avond); unimodaal
weekendprofiel.

**Centrum: ❌ VERWORPEN**
Het centrum vertoont een *trimodaal* weekdagprofiel met pieken op
9u (46,4%), 12u (45,3%) en 18u (43,0%). Dit wijkt af van de
verwachte bimodaliteit en is verklaarbaar door de gemengde
functie van centrumparkings: winkelbezoek (middagpiek) naast
woon-werkverkeer (ochtend/avond). Zhang et al. (2020) en
Channamallu et al. (2024) rapporteren bimodaliteit voor
voornamelijk woon-werkgerichte parkings — de Mechelse context
met een sterk retail-centrum introduceert een bijkomende
middagpiek die in de internationale literatuur onderbelicht
blijft.

**Vesten_of_rand: ✅ BEVESTIGD**
Bimodaal weekdagprofiel (13u en 19u) met één weekendpiek —
consistent met de verwachting voor randparkings met
transitfunctie.

> *Kritische noot:* De definitie van "bimodaal" in de
> toetscode is kwetsbaar: de grens tussen bi- en trimodaal
> hangt af van de piekdetectie-drempel. Rapporteer in de
> thesis dat "het centrum een atypisch retailgedreven profiel
> vertoont dat afwijkt van de woon-werkparadigma's in de
> literatuur" (Channamallu et al., 2024).

#### H-T2 — Autocorrelatie en cyclische structuur
*Verwachting:* Significante ACF-pieken op lag 24h en lag 168h.

**Beide tiers: ✅ BEVESTIGD**
- Centrum: lag24 = −0.378 (sig.), lag168 = +0.345 (sig.)
- Vesten: lag24 = −0.384 (sig.), lag168 = +0.241 (sig.)

ACF lag-24 meet de correlatie van bezetting op tijdstip t met tijdstip t − 24 uur — dus maandag 9u versus zondag 9u, of dinsdag 15u versus maandag 15u. Een negatieve waarde betekent dat wanneer de bezetting op een bepaald uur vandaag hoog is, ze op hetzelfde uur gisteren laag was — en omgekeerd. Dat is niet het verschil tussen ochtend en nacht; dat is een dag-op-dag anticorrelatie bij gelijkblijvend uur. Een plausibeler mechanisme: drukke werkdagmomenten volgen op rustige weekendmomenten (en vice versa), wat over de gehele dataset een negatieve netto-correlatie bij lag-24 genereert. Dat is principieel een weekdagstructuureffect, geen dag/nacht-cyclus. Dit onderscheid is klein maar in de thesis-tekst merkbaar als promotoren ernaar vragen. (Fokker et al., 2021; Wan et al., 2023).

> *Kritische noot:* De 2021–2022 MNAR-gap (13 maanden) vervormt
> de ACF bij hoge lags (> lag 100). De lag-168 bevinding is
> robuust omdat de gap groter is dan 168 uur, maar rapporteer
> dit als limitatie bij de autocorrelatie-interpretatie.

#### H-T3 — Stationariteit
*Verwachting:* Niet-stationair ruw; stationair na Δ₂₄.

**Beide tiers: ✅ BEVESTIGD**
Ruwe reeks: trend-stationair (differentiatie aanbevolen).
Na lag-24 differentiatie: ADF p = 0.0 → stationair.
Dit verantwoordt een seizoenscomponent in de SARIMA-baseline
(nb09) en bevestigt dat lag-features de tijdreeksstructuur
adequaat modelleren (Box & Jenkins, 1976).

> *Kritische noot:* Standaard ADF is conservatief bij
> structurele breuken (2021–2022 gap). De p = 0.0 is hier
> echter dermate sterk dat de conclusie robuust is.

#### H-T4 — COVID-niveauverschil 2020 vs. 2023/2024
*Verwachting:* 2020 toont een lager bezettingsniveau maar
structureel vergelijkbare temporele patronen.

**Centrum: ❌ VERWORPEN (deels)**
Δ = −5,7% (lager in 2020, zoals verwacht ✅), maar
r_struct = −0.956 wijst op een *structureel afwijkend* patroon
— niet enkel een niveauverschil. Dit is problematisch: het
betekent dat 2020-data niet louter als "gecorrigeerde versie"
van 2023/2024 kan worden beschouwd.

**Vesten_of_rand: ❌ VERWORPEN (omgekeerd)**
Δ = +18,6%: vesten_of_rand was in 2020 *hoger* bezet dan in
2023/2024 (MW p ≈ 0, effect medium). Dit is contra-intuïtief
en suggereert een modaliteitsverschuiving post-COVID: na de
pandemie namen centrumparkings meer bezoekers op ten koste
van de randparkings, mogelijk door herstel van het
winkelcentrum en veranderd parkeergedrag (Niu et al., 2023).

> ⚠ *Methodologische implicatie voor nb08:* De year_dummy
> mag NIET ordinaal worden gecodeerd (2020=0, 2023=1,
> 2024=2). Er is geen monotone trend. Gebruik een categorische
> dummy: 2020=0 (pre-herstel), 2023/2024=1 (post-herstel).
> Dit is een directe correctie op de initiële
> feature-shortlist die "ordinaal" suggereerde.


####Belangrijke opmerking !!
Maar hier zit een onbenoemd kernprobleem. Als r_struct = −0.956 voor het centrum — een bijna perfecte negatieve structurele correlatie tussen 2020-patronen en 2023/2024-patronen — dan is de vraag niet enkel hoe te coderen, maar of 2020-data überhaupt in de trainingsset thuis hoort.

Een model dat op 2020-data traint, leert COVID-specifieke gedragspatronen (lockdowns, horeca gesloten, thuiswerk verplicht, winkels aan capaciteitsbeperkingen onderhevig) die in 2025 fundamenteel afwezig zijn. Als die patronen structureel tegengesteld zijn aan 2023/2024, kan het model bij 2025-voorspellingen actief worden gehinderd door de 2020-rijen — niet geholpen. De year_dummy als correctie mitigeert dit gedeeltelijk, maar lost het structurele probleem niet op.

Aanbevolen actie voor nb08 en nb09: Voer een gevoeligheidsanalyse uit waarbij je het model traint met en zonder 2020-data, en vergelijk de validatiescore op 2024. Als het model zonder 2020-data beter valideert op 2024, dan is 2020 contraproductief en moet je dit expliciet rapporteren als databeslissing, met motivatie, in de thesis.

### 2.2 Implicaties nb05 → nb08

| Feature | Encoding | Prioriteit | Motivatie |
|---|---|---|---|
| occupancy_lag_1h | Raw float [0,1] | 🔴 HOOG | ACF lag-1 sig. (H-T2) |
| occupancy_lag_24h | Raw float [0,1] | 🔴 HOOG | ACF lag-24 sig. (H-T2) |
| occupancy_lag_168h | Raw float [0,1] | 🔴 HOOG | ACF lag-168 sig. (H-T2) |
| hour_sin + hour_cos | sin/cos(2π·h/24) | 🔴 HOOG | Cyclische dagstructuur (H-T1/T2) |
| day_type_3 | One-hot | 🔴 HOOG | Weekdag vs. weekend (H-T1) |
| tier | One-hot / stratificatie | 🔴 HOOG | Structureel tiersverschil (H-T1) |
| year_dummy | Categorisch (2020=0, anders=1) | 🟡 MEDIUM | COVID-shift, niet ordinaal (H-T4) |
| month_sin + month_cos | sin/cos(2π·m/12) | 🟡 MEDIUM | Seizoenspatroon (H-T1/cel04) |

---

## 3. nb06 — Ruimtelijke EDA per Parking/Tier

### 3.1 Hypotheseresultaten

#### H-S1 — Centrum hogere bezetting
*Verwachting:* Centrum > vesten/rand in mediaan én piekbezetting
(KW p < 0.01).

**⚠ GEDEELTELIJK**
KW H = 5140.13, p ≈ 0 ✅ — het verschil is statistisch
ondubbelzinnig. Centrum gemiddeld 40,5% vs. vesten 24,4%
(Δ = +16,1%). P90 centrum = 77,5% vs. vesten = 49,1%.
*Maar:* η² = 0.040 en rank-biseriale r = 0.085 — beide
klein. Tier verklaart slechts ~4% van de totale variantie
in bezetting. Het statistisch significante resultaat is
gedeeltelijk een artefact van de grote steekproef (n > 100k).

> ⚠ *Kritische noot:* Statistisch significantie ≠ praktische
> relevantie. Bij n = 129.837 detecteert een KW-toets elk
> triviaal effect. Effect sizes (η², r) zijn de
> informatievere maat (Cohen, 1988; Field, 2013). Rapporteer
> η² primair in de thesis.


##### Effect van statistisch versus praktisch significantie (H-S1)

Interpretatie: De kritische noot over η² = 0.040 is methodologisch correct. Het onderscheid statistisch vs. praktisch significantie is een niveauverschil dat de thesis versterkt.

Ontbrekende nuance: Het Δ = 16,1% niveauverschil (40,5% vs. 24,4%) is voor een parkeerbeleidssimulatie in nb13 wél praktisch relevant — een parking van 500 plaatsen die gemiddeld 40,5% vs. 24,4% bezet is, verschilt met ~80 voertuigen permanent. Dat is relevant voor dynamische tariefstelling. De thesis moet hier expliciet het onderscheid maken: "Voor statistische inferentie is het effect klein (η² = 0.040); voor operationeel parkeerbeheer is het absolute niveauverschil van 16,1% wel degelijk beleidsrelevant." Die dubbele lens ontbreekt momenteel.

#### H-S2 — Capaciteitsparadox
*Verwachting:* Spearman ρ(capacity, mean_occ) < −0.3, p < 0.05.

**⚠ GEDEELTELIJK (richting correct, niet significant)**
ρ = −0.109 (negatieve richting ✅), maar p = 0.764 bij N = 10.

> ⚠ *Methodologische noot:* Dit is het klassieke
> small-N-probleem. Met 10 parkings heeft Spearman ρ een
> power van < 20% om een medium effect te detecteren
> (Cohen, 1992). De bevinding is exploratief en mag niet
> als bewijs of weerlegging worden gepresenteerd. Rapporteer
> als: *"De capaciteitsparadox toont de verwachte negatieve
> richting (ρ = −0.11) maar kan niet statistisch worden
> bevestigd bij N = 10 (p = 0.76)"* (Sun et al., 2023).

##### Belangrijke opmerking
Met N = 10 is dit een correlatietoets met bijna geen statistische power. De wetenschappelijke bijdrage van deze hypothese als formele toets is zo goed als nul. In de thesis kan je beter H-S2 herformuleren als een beschrijvende observatie: "De verwachte negatieve relatie (grotere parking = lagere gemiddelde bezetting) toont de verwachte richting (ρ = −0.11) maar kan met N = 10 niet statistisch worden bevestigd. Dit resultaat is informatief als exploratieve indicatie, niet als bewijs." Zo vermijd je dat een lezer de toets als zinvol beschouwt terwijl ze dat structureel niet is.

#### H-S3 — Intra > inter-tier correlatie
*Verwachting:* r_intra > 0.6; r_inter < 0.4; Fischer z sig.

**⚠ GEDEELTELIJK**
- r_intra = 0.413 (< 0.6 ❌)
- r_inter = 0.323 (< 0.4 ✅)
- Fischer z: t = 1.23, p = 0.227 (n.s. ❌)
- Ward-clustering recapituleert *wel* de tierstructuur visueel.

Het verschil (0.413 vs. 0.323) is aanwezig maar niet
significant en kleiner dan verwacht. Ward-clustering
ondersteunt de tierindeling visueel, maar de correlatiebasis
is zwak. De drempel van r_intra > 0.6 was te optimistisch
gegeven de heterogeniteit binnen de vesten-groep (P Bruul
retail vs. P Tinel residentieel).

> ⚠ *Implicatie:* Tier-stratificatie is *aanbevolen* maar
> niet statistisch dwingend vanuit H-S3 alleen. De
> gecombineerde evidentie van H-S1 (niveauverschil) + H-S3
> (richtingsverschil correlatie) + theoretische gronden
> (Yang et al., 2019; Zhou et al., 2024) rechtvaardigt
> tier als modelleringsstratificatie — maar met de nuance
> dat parking_id een sterkere predictor zal zijn.

Interpretatie: Correct dat r_intra = 0.413 versus r_inter = 0.323 kleiner is dan verwacht. De verklaring via heterogeniteit binnen vesten (P Bruul retail vs. P Tinel residentieel) is inzichtelijk.

##### Ontbrekende consequentie: 
Als de vesten-tier intern zo heterogeen is dat de drempel van r_intra > 0.6 niet wordt gehaald, rijst de vraag of de administratieve tierindeling (centrum vs. vesten/rand) überhaupt de meest informatieve clustering is voor dit model. Een data-gedreven clusteranalyse (k-means of Ward-hiërarchisch op gedragspatronen per parking) had kunnen testen of een betere ruimtelijke indeling bestaat. Dit is niet overwogen in de notebooks. Voor de thesis is het voldoende om dit te vermelden als een limitatie én als aanbeveling voor vervolgonderzoek: "De administratieve tierindeling reflecteert niet noodzakelijk de empirisch meest coherente gedragsclusters, wat data-gedreven herclassificatie tot een relevante aanbeveling voor vervolgonderzoek maakt."


#### H-S4 — Tier-specifiek eventeffect ⭐ Kernhypothese
*Verwachting:* r_rb(centrum) > r_rb(vesten) voor events;
Fischer z significant.

**❌ VERWORPEN — met kritische nuancering**

| Event-type | Δr_rb | p (Fischer z) | Status |
|---|---|---|---|
| Alle events | −0.007 | 0.632 | ❌ |
| Voetbal | −0.374 | 1.000 | ❌ (omgekeerd!) |
| Festival | −0.309 | 0.992 | ❌ (omgekeerd!) |
| **Kermis** | **+0.050** | **0.011** | **✅** |
| Carnaval | −0.194 | 0.991 | ❌ (omgekeerd!) |

Het meest opvallende en zorgwekkende resultaat: voetbal en
festival tonen een *omgekeerd* effect — vesten_of_rand
reageert sterker op deze events dan centrum. Dit is
counter-intuïtief maar verklaarbaar: het Mechelse stadion
(AFAS Stadion) ligt nabij de vestenparkings, waardoor
voetbalevenementen de vesten-bezetting meer opkrikken.
Kermis — het enige bevestigde event-type — betreft een
evenement dat zich in het stadscentrum afspeelt en logisch
een centrumscore oplevert.

> ⚠ *Methodologische implicatie:* Het aggregeren van
> event-types tot één "is_event_day" dummy maskeert
> tegengestelde tier-effecten. Dit is een serieuze
> modelleeringsfout als men event-type niet differentieert.
> In nb08 zijn aparte dummies per event-type verplicht
> (Fokker et al., 2021).

> ⚠ *Consequentie voor de thesis-architectuur:* Zoals
> besproken in de brainstormsessie, is H-S4-falsificatie
> geen doodspoor. De rechtvaardiging voor 2-level SHAP
> steunt op H-S1+S2+S3 gecombineerd. Als H-S4 niet
> universeel geldt, verschuift de thesis-claim van
> "events zijn tier-specifiek" naar "externe
> factor-effecten zijn event-type-specifiek en geografisch
> bepaald" — een even waardevolle bijdrage die beter
> aansluit bij de werkelijke data.

Interpretatie: De herformulering ("event-effect is type- en geografisch specifiek, niet universeel tier-specifiek") is methodologisch het sterkste stuk redenering in de volledige EDA. ✅

##### Extra kritisch punt: 
de steekproefgrootte per event-type. Als er in 3 jaar trainingsdata slechts 15–20 voetbalmatchen zijn (KV Mechelen speelt ~17 thuiswedstrijden per seizoen; de 2021-2022 gap valt weg), dan is de statistische basis voor een dedicated voetbal-dummy extreem smal. Aparte dummies per event-type (zoals aanbevolen in nb08) zijn alleen zinvol als elke klasse voldoende voorkomens heeft voor stabiele schatting — een vuistregel is minimaal 30–50 positieve observaties per klasse in een ML-context. Dit moet worden geverifieerd vóór nb08 de feature-shortlist implementeert. Anders zijn de event-type dummies overfitting-kandidaten die op de holdout (2025) dramatisch degraderen.

### 3.2 Implicaties nb06 → nb08

| Feature | Encoding | Prioriteit | Motivatie |
|---|---|---|---|
| parking_id | One-hot (10) of target encoding | 🔴 HOOG | Sterkste heterogeniteit (H-S1) |
| tier | One-hot / stratificatie | 🔴 HOOG | Structureel niveauverschil (H-S1/S3) |
| event_type_dummies | Apart per type | 🔴 HOOG | Omgekeerde H-S4-effecten per type |
| tier × event_type | Interactie product | 🟡 MEDIUM | Type-specifiek tiereffect (H-S4) |
| total_capacity | log-getransformeerd | 🟢 LAAG | ρ = −0.109, N=10 (H-S2 zwak) |

---

## 4. nb07 — EDA Externe Factoren

### 4.1 Hypotheseresultaten

#### H-E1 — Niet-lineair neerslageffect ✅ BEVESTIGD
KW significant (p < 0.01) over neerslagbins. De kwadratische
OLS β₂ is niet significant, wat suggereert dat de
niet-lineariteit eerder een *drempel*- dan een paraboolpatroon
volgt — consistent met Oz (2023) die drempelgedrag bij
~2 mm/u identificeert. Binned encoding (droog/licht/matig/
zwaar) is dan ook de correcte nb08-aanpak boven een
kwadratisch polynoom.

Interpretatie: Correct dat een drempelpatroon passender is dan een kwadratisch polynoom.

Ontbrekend: De keuze van de bingrenzend is nergens gemotiveerd of gedocumenteerd. Zijn ze gebaseerd op meteorologische standaardcategorieën (KNMI/RMI)? Of arbitrair? In de thesis moet de herkomst van de bins worden verantwoord; anders is de binned encoding onreproduceerbaar. "De neerslagcategorieën (droog/licht/matig/zwaar) zijn gedefinieerd op basis van [bron], conform de drempelanalyse van Oz (2023)."

#### H-E2 — Seizoensgebonden temperatuureffect ❌ VERWORPEN
ρ_winter = +0.065, ρ_zomer = +0.176 — beide positief, geen
tekenwisseling. Dit weerlegt de verwachte winter-positief /
zomer-negatief-structuur uit Balmer et al. (2021) en
Correia et al. (2020).

> ⚠ *Interpretatie:* In Mechelen correleert hogere
> temperatuur in *alle* seizoenen positief met
> parkeeroccupantie. Een mogelijke verklaring is dat
> Mechelen als retail-stad een seizoenspatroon heeft waarbij
> warm weer meer winkelbezoek stimuleert — ongeacht het
> seizoen. Dit kan ook een confoundingeffect zijn:
> temperatuur en weekdag/feestdag zijn gecorreleerd
> (warme zomerdagen = meer uitstappen). De interactieterm
> temp_c × seizoen is desondanks *aanbevolen* in nb08 omdat
> de absolute effectgrootte per seizoen kan verschillen,
> ook zonder tekenwisseling (Balmer et al., 2021).

Interpretatie: De positieve correlatie in alle seizoenen is correct gerapporteerd. De retail-verklaring is plausibel maar speculatief.

#### Alternatieve verklaring: 
Temperatuur in België correleert sterk met dagduur (uren licht). Langere dagen = meer uren waarop winkels open en bezoekers actief zijn = hogere gemiddelde dagbezetting, ongeacht gedragseffecten. Dit is een confound die de positieve ρ (in principe) volledig kan verklaren zonder dat er sprake is van direct gedragseffect van temperatuur. Het opnemen van sun_duration_min in de multivariate analyse had dit kunnen onderscheiden — maar H-E7 toont dat zon en temperatuur te collineair zijn om te scheiden. Vermeld dit als een fundamentele interpretatiegrens: "De positieve temperatuur-correlatie over alle seizoenen is mogelijk mediërend via daglengte, niet direct via gedrag. Causale interpretatie is op basis van deze data niet mogelijk."


#### H-E3 — Cascade event-lageffect ✅ BEVESTIGD
t = −1 significant ✅; t = +1 significant ✅. Het
pre-event cascade-profiel is empirisch aangetoond: bezoekers
arriveren aantoonbaar vóór de officiële eventstart en
vertrekken gefaseerd na afloop. Dit verantwoordt directe
lag-features in nb08: `hours_to_next_event` en
`hours_since_last_event` (Fokker et al., 2021).


Interpretatie, conclusie en consequentie zijn correct en volledig. De cascade-lag features (hours_to_next_event, hours_since_last_event) zijn goed gemotiveerd.

##### Aandachtspunt: 
De definitie van "event" is niet geëxpliciteerd. Worden kleine buurtmarktjes meegeteld? Alleen events boven een bepaalde schaal? Gezien H-E9 toont dat 53% van de events confidence="estimated" heeft, is het relevant te vermelden dat het cascade-effect is aangetoond op basis van alle events (inclusief onzekere data) of enkel geverifieerde events.


#### H-E4 — Tegengesteld feestdageffect per tier ❌ VERWORPEN
Noot: Dit is de tweede keer dat p = 1.000 opduikt — ook bij H-S4 voetbal. Een geïsoleerde p=1.000 is een implementatiebug. Twee keer in dezelfde codebase suggereert een systematische fout in de Fischer z-implementatie: waarschijnlijk een deling door nul of een randgeval waarbij twee r-waarden identiek zijn, waarna z = 0 en p = 1.000. Dit invalidiseert niet alleen H-E4, maar roept ook twijfel op over de betrouwbaarheid van alle andere Fischer z-resultaten in de notebooks (H-S4, H-E3, H-E8). 

Δr_rb = −0.148, p = 1.000. Het verwachte tegengestelde
effect (centrum ↓, vesten ↑) is niet aangetoond. Mogelijke
verklaring: nationale feestdagen in Mechelen brengen meer
winkelbezoek naar het centrum (winkelend publiek geniet van
de vrije dag), waardoor het verwachte "gesloten winkels
= minder bezoekers"-effect niet optreedt of gemaskeerd wordt.

> ⚠ *Opmerking:* p = 1.000 is een red flag — dit kan
> wijzen op een implementatiefout in de Fischer z-toets
> (mogelijke deling door nul of extreme n-waarden).
> Controleer de onderliggende r_rb-waarden en steekproef-
> groottes per tier voor feestdagen expliciet in nb08.

#### H-E5 — VIF multicollineariteit ❌ VERWORPEN
0 features met VIF > 5, zowel raw als cyclisch. De
verwachte multicollineariteit tussen temp_c en month treedt
niet op in deze dataset bij de huidige feature-set.

> ⚠ *Methodologische noot:* Dit is een opmerkelijk
> resultaat. Temperatuur en maand zijn in de meeste
> klimaatdatasets sterk gecorreleerd (r > 0.7). Mogelijke
> verklaring: de VIF-berekening is uitgevoerd op een
> gestratificeerde steekproef (n=20.000) waarbij de
> cyclische features (sin/cos) orthogonaler zijn dan de
> ruwe maandwaarden. De aanbeveling voor cyclische
> encoding blijft geldig als best practice (Cerqueira
> et al., 2023), ook al dwingt de VIF-toets ze niet af
> in deze specifieke dataset.



#### Aanvulling (H-E5)
Interpretatie: De noot dat de interactieterm temp_c × seizoen mogelijk ontbreekt in de VIF-berekening is correct en cruciaal.

Hetzelfde geldt voor lag-features. occ_lag_1h en occ_lag_24h zijn onderling gecorreleerd (een parking die een uur geleden druk was, was vaak ook gisteren op dit uur druk). Als de VIF-berekening geen lagfeatures bevat, is ze onvolledig. Het resultaat "0 features boven VIF=5" is daarmee onzeker. Dit is geen reden om cyclische encoding te verwerpen — die best practice staat los van de VIF-uitkomst — maar het resultaat mag niet worden gepresenteerd als bewijs van multicollineariteitsafwezigheid in de volledige feature-set.

#### H-E6 — Winddrempeleffect ✅ BEVESTIGD
Δ mediaan = +5,3% bij wind > 10 m/s; MW p = 0.036. Het
drempeleffect is significant maar het effect is klein.
Bij windsnelheden > 10 m/s is parkeeroccupantie geassocieerd
met een hogere bezetting, consistent met modal shift van
fiets naar auto (Böcker et al., 2013).

> ⚠ *Kanttekening:* n bij wind > 10 m/s is beperkt in
> Mechelen (continentaal klimaat). De toets heeft lage
> power. Rapporteer als exploratieve bevinding; bevestiging
> via SHAP in nb12 is wenselijk.

#### Kritiek (H-E6)
Interpretatie en conclusie zijn proportioneel.  De drempelkeuze van 10 m/s is echter nergens gemotiveerd. Was dit data-gedreven (percentielgrens) of literatuurgedreven (Böcker et al., 2013 gebruikt een andere drempel)? Kleine punt, maar nodig voor reproduceerbaarheid.

#### H-E7 — Negatief zonne-effect in zomer ❌ VERWORPEN
Partiële ρ gecorrigeerd voor temp_c toont geen significant
negatief zomer-effect. Zonneschijnduur heeft in Mechelen
geen aantoonbare substitutie-relatie met parkeergedrag,
mogelijk omdat zon en temperatuur zo sterk collineair zijn
dat het partiële effect wegvalt.

#### Ontbrekende confound (H-E7): 
Belgische schoolvakanties vallen structureel samen met de periodes met de meeste stedelijke evenementen: kerstmarkt (kerstvakantie), carnaval (krokusvakantie), festivals en kermissen (zomervakantie). Het gemeten schoolvakantie-effect is dus gedeeltelijk een event-effect, en gedeeltelijk een seizoenseffect. In de thesis mag je dit niet presenteren als een zuiver "schoolvakantie-effect" zonder deze confound te vermelden. In het model (nb09+) zullen event-dummies en schoolvakantie-dummies samen worden opgenomen — dan wordt de partiële bijdrage automatisch geschat. Maar de EDA-conclusie overstates de causale kracht van schoolvakanties per se.

#### H-E8 — Schoolvakanties tier-specifiek ✅ BEVESTIGD
Δr_rb = +0.117, p = 0.000 ✅. Schoolvakanties verlagen
bezetting centrum én verhogen relatief bezetting
vesten_of_rand — consistent met de verwachting. Dit is
het meest robuuste kalendereffect in nb07, met een
significant Fischer z-resultaat.

> Dit is een sterke bevinding voor nb08: de interactie
> `tier × is_school_vacation` is een goed onderbouwde
> feature-kandidaat (Zhang et al., 2024).

#### H-E9 — Schaalafhankelijk eventeffect ❌ VERWORPEN
Geen significante Spearman ρ tussen event_scale_max en
piek-Δ. Dit kan structureel zijn (kleine events zijn in
Mechelen net zo impactvol als grote) of een
datakwaliteitsprobleem weerspiegelen: 185 van 346 event-
dagen hebben confidence = "estimated", waardoor de
schaalindeling inherent onbetrouwbaar is.

### 4.2 RF Proxy Feature Importance (cel 11)

De Random Forest permutation importance geeft een
niet-parametrisch inzicht in feature-relevantie:

| Tier | Top-3 features |
|---|---|
| Centrum | total_capacity, year_dummy, hour_cos |
| Vesten_of_rand | hour_cos, wind_speed_ms, sun_duration_min |

> ⚠ *Opvallend:* `total_capacity` staat bovenaan voor
> centrum — dit is een *statische* feature die per
> parking_id constant is. De RF leert hier feitelijk
> een parking_id-proxy. Dit bevestigt dat parking_id
> de dominante ruimtelijke predictor is, niet tier.
> In nb08 moet worden gecontroleerd of total_capacity
> en parking_id niet redundant zijn (multicollineariteit
> via VIF of RF importance met en zonder parking_id).

---

## 5. Geconsolideerde hypothesescore

| Code | Hypothese | Status | Kritisch? |
|---|---|---|---|
| H-T1 | Bimodaal dagprofiel | ⚠ Tier-specifiek | ➕ Trimodaal centrum = nieuwe bevinding |
| H-T2 | ACF-pieken lag24/168 | ✅ Bevestigd | — |
| H-T3 | Stationair na Δ₂₄ | ✅ Bevestigd | — |
| H-T4 | COVID niveaushift | ❌ Verworpen | ➕ Omgekeerd voor vesten |
| H-S1 | Centrum hogere bezetting | ⚠ Gedeeltelijk | η² klein |
| H-S2 | Capaciteitsparadox | ⚠ Gedeeltelijk | N=10 power-probleem |
| H-S3 | Intra > inter-tier correlatie | ⚠ Gedeeltelijk | n.s., maar richting correct |
| H-S4 | Eventeffect tier-specifiek | ❌ Verworpen | ➕ Kermis wél sig. |
| H-E1 | Niet-lineair neerslageffect | ✅ Bevestigd | — |
| H-E2 | Seizoensgebonden temp | ❌ Verworpen | Positief alle seizoenen |
| H-E3 | Cascade event-lag | ✅ Bevestigd | — |
| H-E4 | Feestdag tier-tegengesteld | ❌ Verworpen | p=1.000 check impl. |
| H-E5 | VIF multicollineariteit | ❌ Verworpen | Cyclisch encoding best practice |
| H-E6 | Winddrempeleffect | ✅ Bevestigd | Klein effect |
| H-E7 | Zon negatief zomer | ❌ Verworpen | Collineariteit met temp |
| H-E8 | Schoolvakantie tier-specifiek | ✅ Bevestigd | Sterkste kalendereffect |
| H-E9 | Schaalafhankelijk event | ❌ Verworpen | Datakwaliteit probleem |

**Score: 6× bevestigd · 7× verworpen · 4× gedeeltelijk**

---

## 6. Master feature-shortlist voor nb08

### 🔴 PRIORITEIT HOOG — opnemen zonder discussie

| Feature | Encoding | Basis |
|---|---|---|
| occupancy_lag_1h / 24h / 168h | Raw float | H-T2: ACF |
| hour_sin + hour_cos | sin/cos(2π·h/24) | H-T1/T2 |
| day_type_3 | One-hot (3 klassen) | H-T1 |
| parking_id | One-hot of target encoding | H-S1: dominante predictor |
| tier | One-hot of stratificatie | H-S1/S3 |
| precip_bin | 4 bins (droog/licht/matig/zwaar) | H-E1 |
| is_school_vacation + tier × interactie | Binair + product | H-E8 |
| event_type_dummies (5 typen) | Aparte binaire dummy | H-S4 (type-specifiek) |
| year_dummy | Categorisch (2020=0, anders=1) | H-T4 (niet ordinaal!) |

### 🟡 PRIORITEIT MEDIUM — testen in nb08

| Feature | Encoding | Basis |
|---|---|---|
| month_sin + month_cos | sin/cos(2π·m/12) | H-T1 seizoen |
| temp_c | Gestandaardiseerd + seizoen-interactie | H-E2 (positief, geen tekenwisseling) |
| wind_speed_ms of dummy >10m/s | Drempel of continu | H-E6 |
| is_national_holiday + tier × interactie | Binair + product | H-E4 (check impl.) |
| weekday_sin + weekday_cos | sin/cos(2π·wd/7) | H-T2 |
| hours_to_next_event | Continu (lag-feature) | H-E3 cascade |
| hours_since_last_event | Continu (lag-feature) | H-E3 cascade |

### 🟢 PRIORITEIT LAAG — optioneel / appendix

| Feature | Encoding | Basis |
|---|---|---|
| sun_duration_min | Gestandaardiseerd | H-E7 verworpen |
| total_capacity | log-getransformeerd | H-S2 zwak, redundant met parking_id |
| n_concurrent_events | Continu | H-E10 (niet formeel getoetst) |

---

## 7. Databeperkingen — verplichte vermeldingen in thesis

1. **P Zandpoortvest (622 pl.)** — aanwezig in locatietabel, absent
   in tijdreeksdata. De vesten_of_rand-tier onderschat de werkelijke
   capaciteit met ~79%. Alle tier-uitspraken voor vesten_of_rand zijn
   *conservatief*.

2. **2021–2022 MNAR-gap** — 13 maanden systeemblackout. Niet willekeurig
   (MNAR), niet bruikbaar als trainingsdata. Veroorzaakt een structurele
   breuk in autocorrelatieplots bij hoge lags.

3. **N = 10 parkings** — statistische power voor ruimtelijke hypothesen
   (H-S2, H-S3) is fundamenteel beperkt. Rapporteer effectgrootte
   primair; p-waarden zijn informatief maar niet leidend (Cohen, 1992).

4. **data_confidence "estimated"** — 185 van 346 event-dagen hebben
   geen geverifieerd tijdstip of schaal. Robuustheidscheck (verified
   only) uitgevoerd in nb07 cel 08; vermeld stabiliteit expliciet.

5. **Off-street context** — de SLR is grotendeels gebaseerd op
   on-street parking studies. Bevindingen zijn niet direct
   vergelijkbaar. Nieuwe SLR (SciSpace, off-street focus) is
   geïnitieerd om deze gap te dichten.

6. **DST-anomalieën** — 36 uur per jaar (zomer/wintertijdovergang)
   gedocumenteerd; al gefilterd via de trainingsfilter.

---

## 8. Kritische methodologische opmerkingen

> **⚠ p=1.000 bij H-E4 (Fischer z)** — Een exacte p=1.000 is
> statistisch onmogelijk bij een tweezijdige toets tenzij z=0 of
> er een implementatiefout is (deling door nul, lege subgroep,
> of identieke r_rb-waarden). Controleer vóór nb08 de
> onderliggende r_rb-waarden en steekproefgrootte per tier op
> nationale feestdagen.

> **⚠ H-E5 VIF=0 ondanks temp × month collineariteit** — Cyclische
> features zijn orthogonaler dan ruwe integers, wat de lage VIF
> deels verklaart. Maar 0 features boven VIF=5 bij de volledige
> kandidaat-set (inclusief temp_c en seizoen) is verrassend.
> Hercontroleer of de VIF-berekening de interactieterm
> temp_c × seizoen heeft meegenomen — die is doorgaans de
> probleemterm.

> **⚠ year_dummy ordinaal in shortlist** — De nb07-scorekaart
> vermeldt "Ordinaal (2020=0, 2023=1, 2024=2)" maar H-T4 toont
> expliciet dat er geen monotone trend is. Correcte encoding is
> categorisch: 2020=0, 2023/2024=1. Dit is een interne
> inconsistentie die in nb08 moet worden rechtgezet.

> **⚠ tier × is_event_day in shortlist ondanks H-S4 falsificatie**
> — De shortlist behoudt terecht deze interactieterm, maar de
> motivatie moet worden bijgesteld: niet "eventeffect groter voor
> centrum" maar "event-effect is tier- én type-specifiek, waarbij
> de richting per event-type verschilt". De feature is zinvol als
> de coëfficiënt vrij kan variëren per combinatie.

---

## 9. Wat leert FASE 02 ons voor nb08 en verder?

### Architectuurkeuze bevestigd
Optie A (één model + tier als feature + SHAP per tier-subset)
wordt door de data gesteund. De zwakke H-S3-bevestiging en de
type-specifieke H-S4-patronen wijzen erop dat parkings
heterogener zijn dan tiers — parking_id is de sterkste
ruimtelijke predictor.

### Verrassende bevindingen die de thesis verrijken
- Het centrum heeft een *trimodaal* profiel door retail-gedrag —
  uniek t.o.v. internationale literatuur
- Voetbalevenementen verhogen bezetting vesten meer dan centrum —
  geografisch verklaarbaar door stadionlocatie
- COVID-effect is tier-asymmetrisch: vesten steeg in 2020,
  centrum daalde — wijst op fundamenteel ander gebruikersgedrag
  per tier

### Prioriteit voor nb08
1. **Lag-features** (occ_lag_1h/24h/168h + event-lag-features) —
   sterkste theorie- én empirisch-gedreven keuze
2. **parking_id** — dominant boven tier als ruimtelijke predictor
3. **event_type_dummies apart** — aggregatie maskeert
   tegengestelde effecten
4. **precip_bin** — niet-lineair effect bevestigd
5. **is_school_vacation + tier-interactie** — sterkste
   kalendereffect

### Risico voor nb12 (SHAP)
Als parking_id de dominante predictor is, zullen SHAP-waarden
per tier minder informatief zijn dan SHAP-waarden per parking.
Overweeg een derde SHAP-niveau (parking-level) naast global
en tier — of rapporteer parking-level in de appendix.

---

*Einde FASE 02 Recap — versie 1.0 | 2026-03-04*
*Volgende stap: nb08 — Feature Engineering Pipeline*


## Eerlijke inventaris: waar staan we?

***

## Het grote plaatje: solide basis, maar twee structurele spanningspunten

De data is rijk genoeg voor een degelijke masterproef. Maar twee bevindingen uit de EDA creëren echte spanningspunten die nu — vóór nb08 — moeten worden benoemd en verwerkt, anders komen ze je later als verrassing achterna.

***

## Wat sterk staat ✅

**De tijdreeksstructuur is ijzersterk.**
H-T2 en H-T3 zijn ondubbelzinnig bevestigd. Lag-features (occ_lag_1h/24h/168h) en cyclische encoding zijn empirisch gefundeerd. Dit is het ruggengraat van elk goed parkeermodel — en dat fundament klopt (Fokker et al., 2021; Wan et al., 2023).

**Externe factoren werken.** H-E1 (neerslag), H-E3 (cascade), H-E6 (wind), H-E8 (schoolvakanties) zijn allemaal bevestigd. RQ1 — *"verbeteren externe factoren de voorspelling?"* — zal met hoge waarschijnlijkheid een positief antwoord krijgen. Dat is de kern van de thesis.

**De onverwachte bevindingen zijn academisch waardevol.**
Het trimodale centrumprofiel, de omgekeerde COVID-asymmetrie per tier, en het voetbaleffect dat vesten meer raakt dan centrum — dit zijn originele observaties die *niet* in de internationale literatuur staan. Mechelen als case is academisch interessant juist omdat het afwijkt.

**De trainingsdata is kwalitatief in orde.**
129.837 gefilterde rijen, 3 aaneengesloten jaren (met bekende gap), een schone holdout in 2025. Voor een ML-model is dit werkbaar.

***

## De twee structurele spanningspunten ⚠

### Spanningspunt 1: H-S4 is verworpen — de kernthese moet worden hergeformuleerd

Dit is het grootste risico. De originele kernboodschap luidde:

> *"Externe factoren werken anders per tier, en events zijn dominanter in centrum."*

De data zegt: **nee, dat klopt niet universeel.** Voetbal en festivals raken vesten meer. Alleen kermis toont het verwachte centrumeffect.

**Maar dit is oplosbaar.** De herformulering die de data wél ondersteunt:

> *"Externe factoren werken anders per parking en per event-type. Tier is een proxy voor ruimtelijke heterogeniteit, maar parking_id is de sterkere predictor. De thesis-bijdrage is het ontmaskeren van welke externe factor wel en niet tier-specifiek is via 2-level SHAP."*

Dit is **een sterkere, nuancerijkere claim** dan de originele. Het is ook eerlijker tegenover de data.

### Spanningspunt 2: Tier als stratificatie is zwakker dan verwacht

H-S3 is gedeeltelijk bevestigd (richting correct, niet significant). De RF importance toont dat `total_capacity` — feitelijk een parking_id-proxy — de dominantste ruimtelijke predictor is, niet tier.

**Implicatie voor nb10–nb12:** Als je aparte modellen per tier traint (Optie B), heb je voor `rand` slechts 1 parking (P Keerdok). Dat is statistisch niet verdedigbaar. Optie A (één model + tier als feature + SHAP per subset) is de enige realistische architectuur.

***

## Inventaris per onderzoeksvraag

### RQ1 — Externe factoren vs. puur temporeel model
**Potentieel: 🟢 HOOG**

De bevestigde externe factoren (neerslag, wind, schoolvakanties, event-cascade) zullen de MAPE/RMSE meetbaar verlagen t.o.v. een puur temporeel model. Dit is in lijn met 30 van 50 SLR-papers. RQ1 is het meest zeker te beantwoorden.

*Risico:* Als het effect klein is (< 2–3% MAPE-verbetering), is de academische bijdrage beperkt. Maar gezien de cascade-bevestiging en schoolvakantie-effect is een zinvolle verbetering realistisch.

### RQ2 — Dynamische prijssimulatie
**Potentieel: 🟡 MEDIUM — conditioneel**

RQ2 hangt volledig af van de kwaliteit van het model uit nb09–nb11. Als de voorspellingen accurate bezettingsprofielen geven, is de what-if simulatie (met theoretische prijselasticiteitscoëfficiënten, Li et al., 2024) verdedigbaar. Maar:

- Geen historische prijsdata → simulatie is theoretisch, niet empirisch
- P Zandpoortvest ontbreekt → spillover-simulaties zijn onderschat
- Slechts 10 parkings → beperkte geografische spreiding voor beleidsconclusies

**Aanbeveling:** Formuleer RQ2 als *"wat-als scenario"* met expliciete assumpties, niet als empirisch bewijs van prijsbeleid. Dit is methodologisch verdedigbaar (Shoup, 2005 dynamische prijstheorie; Li et al., 2024).

### RQ3 — Transferabiliteit naar andere steden
**Potentieel: 🔴 LAAG — herevalueer de ambitie**

Dit is het meest kwetsbare deel van het opzet. Met:
- 1 stad
- 10 parkings
- Geen vergelijkingsdata van andere steden
- Off-street context die nauwelijks gedocumenteerd is in de literatuur

...is een kwantitatieve transferabiliteitsanalyse niet realistisch. Je kunt geen minimum-datavereisten empirisch valideren zonder een tweede stad.

**Mogelijke redding:** Formuleer RQ3 als *"welke modelcomponenten zijn context-onafhankelijk (temporele structuur) vs. context-afhankelijk (parking_id, lokale events)?"* en gebruik de SHAP-resultaten om dit te onderbouwen. Dit is een kwalitatieve transferabiliteitsreflectie — minder ambitieus maar eerlijk en verdedigbaar (Shao et al., 2024 transfer learning; Nezhadettehad et al., 2025).

***

## Realistisch eindperspectief

| Onderdeel | Status | Verwachting |
|---|---|---|
| EDA (nb05–07) | ✅ Volledig | Sterk fundament |
| Feature engineering (nb08) | 🔄 Volgende stap | Duidelijke shortlist beschikbaar |
| Baseline modellen (nb09) | 🔄 Later | SARIMA + XGBoost baseline haalbaar |
| ML/DL modellen (nb10–11) | 🔄 Later | RF/XGBoost meest realistisch voor 10 parkings |
| SHAP 2-level (nb12) | 🔄 Later | Haalbaar met hergeformuleerde H-S4 |
| Prijssimulatie (nb13) | ⚠ Conditioneel | Alleen als model kwaliteit voldoende is |
| Transferabiliteit (nb14) | ⚠ Herevalueren | Kwalitatief i.p.v. kwantitatief |

***

## Concrete aanbeveling voor nu

**Doe één ding vóór nb08:** Herschrijf de kernboodschap van de thesis in twee zinnen die de werkelijke data-bevindingen weerspiegelen. De promotor zal dit vroeg of laat vragen, en de huidige formulering klopt niet meer met de EDA-uitkomsten. Een eerlijk verhaal dat afwijkt van de verwachting is een betere thesis dan een verhaal dat de data forceert in de originele hypothese.

De thesis is haalbaar, onderscheidend, en heeft echte academische waarde. Maar de ambitie van RQ3 moet realistischer worden geformuleerd — liever een scherpe RQ1+RQ2 dan een dunne RQ3 die de jury kritisch bevraagt.